# Customer Churn Prediction Using an ANN: Live Demo

Run the cells from top to bottom. Each section matches a part of the video:
**data, segmentation, architecture, training, evaluation, predictions, drivers, recommendations, limitations.**

In [ ]:
import os, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (silhouette_score, accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay)
pd.options.display.float_format = "{:,.3f}".format
pd.set_option("display.max_colwidth", None)   # show long text in full
pd.set_option("display.width", 200)

def fmt(df, **spec):
    """Return a copy of df with the given columns formatted as text (works on every pandas version)."""
    out = df.copy()
    for col, f in spec.items():
        out[col] = out[col].map(f.format)
    return out

# If the notebook cannot find your files, paste the full path of the folder that contains your 'data' folder
# between the quotes below (forward slashes work on Windows), for example:
#     PROJECT_FOLDER = "C:/Users/yourname/Desktop/Project/Final"
PROJECT_FOLDER = None

# Find files wherever they are saved: next to this notebook, in a 'data' folder, or in the folders above it
def _roots():
    roots = []
    if PROJECT_FOLDER:
        roots.append(Path(PROJECT_FOLDER))
    nb_file = globals().get("__vsc_ipynb_file__")          # VS Code: the path of this notebook (if available)
    if nb_file:
        roots.append(Path(nb_file).resolve().parent)
    roots.append(Path.cwd())
    out = []
    for r in roots:
        for base in [r] + list(r.parents)[:2]:
            if base not in out:
                out.append(base)
    return out

def locate(name, required=True):
    bases = _roots()
    stem, suffix = os.path.splitext(name)
    for target in (name, f"{stem}*{suffix}"):              # exact name first, then names like 'file (1).csv'
        for base in bases:                                 # quick checks: base/data/ and base/
            for sub in ("data", "."):
                try:
                    for p in (base / sub).glob(target):
                        return p
                except OSError:
                    pass
        for base in bases:                                 # then one and two folders further down
            for pattern in (f"*/{target}", f"*/*/{target}"):
                try:
                    for p in base.glob(pattern):
                        return p
                except OSError:
                    pass
    if required:
        looked = "\n  ".join(str(b) for b in bases)
        raise FileNotFoundError(f"Could not find '{name}'. Looked in these folders and the ones inside them:\n  {looked}\n"
                                f"Fix: move this notebook into the folder that holds your 'data' folder, or set PROJECT_FOLDER "
                                f"near the top of this cell to that folder's path, then run the cell again.")
    return None

print("Dataset found in folder:", locate("Dataset_ATS_v2.csv").parent.name)

SEED = 42
TRAIN_LIVE = True   # True = train the ANN live (about a minute). False = load the saved model instantly.
print("Ready. TensorFlow", tf.__version__)

## 1. The data
7,043 telecommunications customers. The goal is to predict who will churn (leave the company).

In [ ]:
raw   = pd.read_csv(locate("Dataset_ATS_v2.csv"))
train = pd.read_csv(locate("churn_train_processed.csv"))
test  = pd.read_csv(locate("churn_test_processed.csv"))
FEATURES = [c for c in train.columns if c != "Churn"]

print(f"Customers: {len(raw):,}   |   churned: {(raw.Churn == 'Yes').mean():.1%}")
print(f"Training set: {len(train):,} customers   |   Test set: {len(test):,} customers   |   model inputs: {len(FEATURES)}")
raw.head()

## 2. Customer segmentation (K-means)
How many groups? Try 2 to 10 clusters and look for the elbow and the best silhouette score.

In [ ]:
scaled = pd.read_csv(locate("churn_clustering_ready.csv"))
Xc = scaled[["tenure", "MonthlyCharges"]].values

ks, inertia, sil = list(range(2, 11)), [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Xc)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(Xc, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(ks, inertia, "o-"); ax[0].set(title="Elbow method", xlabel="Number of clusters (k)", ylabel="Inertia")
ax[1].plot(ks, sil, "o-", color="tab:orange"); ax[1].set(title="Silhouette score", xlabel="Number of clusters (k)", ylabel="Silhouette")
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()
print(f"Best silhouette score at k = {ks[int(np.argmax(sil))]} ({max(sil):.3f})  ->  we use 4 segments")

In [ ]:
km = KMeans(n_clusters=4, n_init=10, random_state=SEED).fit(Xc)
raw["cluster"] = km.labels_
raw["churned"] = (raw.Churn == "Yes").astype(int)

prof = raw.groupby("cluster").agg(customers=("churned", "size"),
                                  avg_tenure_months=("tenure", "mean"),
                                  avg_monthly_charge=("MonthlyCharges", "mean"),
                                  churn_rate=("churned", "mean"),
                                  churners=("churned", "sum"))
t_med, m_med = raw.tenure.median(), raw.MonthlyCharges.median()
prof["segment"] = [("New" if r.avg_tenure_months < t_med else "Loyal") +
                   (" Premium" if r.avg_monthly_charge >= m_med else " Budget") for _, r in prof.iterrows()]
prof.loc[prof.churn_rate.idxmax(), "segment"] += " (high risk)"
prof["share_of_all_churners"] = prof.churners / prof.churners.sum()
seg_name = prof["segment"].to_dict()
raw["segment"] = raw.cluster.map(seg_name)

view = prof.set_index("segment").sort_values("churn_rate", ascending=False).drop(columns="churners")
display(fmt(view, customers="{:,}", avg_tenure_months="{:.1f}", avg_monthly_charge="${:.2f}",
            churn_rate="{:.1%}", share_of_all_churners="{:.1%}"))

colors = dict(zip(sorted(seg_name.values()), ["tab:green", "tab:purple", "tab:blue", "tab:red"]))
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for n, g in raw.groupby("segment"):
    ax[0].scatter(g.tenure, g.MonthlyCharges, s=6, alpha=.4, label=n)
ax[0].set(title="Customer segments", xlabel="Tenure (months)", ylabel="Monthly charges ($)"); ax[0].legend(markerscale=3, fontsize=8)
order = prof.sort_values("churn_rate", ascending=False)
ax[1].bar(order.segment, order.churn_rate * 100, color="tab:red")
ax[1].axhline(raw.churned.mean() * 100, ls="--", c="grey")
for i, v in enumerate(order.churn_rate * 100): ax[1].text(i, v + 1, f"{v:.1f}%", ha="center")
ax[1].set(title="Churn rate by segment (dashed = overall 26.5%)", ylabel="Churn rate (%)"); ax[1].tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()

## 3. ANN architecture
12 inputs, then 16 neurons (ReLU), 8 neurons (ReLU), and 1 sigmoid output giving the probability of churn.

In [ ]:
def build_model(hidden=(16, 8), n_inputs=len(FEATURES)):
    model = keras.Sequential(name="ANN_16_8")
    model.add(layers.Input(shape=(n_inputs,), name="Input_Layer"))
    for i, units in enumerate(hidden, 1):
        model.add(layers.Dense(units, activation="relu", name=f"Hidden_Layer_{i}"))
    model.add(layers.Dense(1, activation="sigmoid", name="Output_Layer"))
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

model = build_model()
model.summary()
diagram = locate("architecture_diagram.png", required=False)
if diagram:
    display(Image(filename=str(diagram), width=560))

## 4. Training
Adam optimiser, binary cross-entropy, batch size 32, up to 50 epochs, 20% of the training data held out for validation, early stopping.

In [ ]:
X_full, y_full = train[FEATURES].values.astype("float32"), train.Churn.values
X_tr, X_val, y_tr, y_val = train_test_split(X_full, y_full, test_size=0.20, stratify=y_full, random_state=SEED)

class Progress(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % 10 == 0:
            print(f"epoch {epoch + 1:>2}/50   loss {logs['loss']:.4f}   val_loss {logs['val_loss']:.4f}   val_accuracy {logs['val_accuracy']:.3f}")

trained_live = False
if TRAIN_LIVE:
    try:
        tf.keras.utils.set_random_seed(SEED)
        model = build_model()
        hist = model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=50, batch_size=32, verbose=0,
                         callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True), Progress()])
        fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
        ax[0].plot(hist.history["accuracy"], label="Training"); ax[0].plot(hist.history["val_accuracy"], label="Validation")
        ax[0].set(title="Model accuracy", xlabel="Epoch", ylabel="Accuracy"); ax[0].legend(); ax[0].grid(alpha=.3)
        ax[1].plot(hist.history["loss"], label="Training"); ax[1].plot(hist.history["val_loss"], label="Validation")
        ax[1].set(title="Model loss", xlabel="Epoch", ylabel="Binary cross-entropy"); ax[1].legend(); ax[1].grid(alpha=.3)
        plt.tight_layout(); plt.show()
        trained_live = True
    except Exception as err:
        print(f"Live training could not run in this session ({type(err).__name__}: {str(err)[:200]}).")
        print("Loading the saved, already-trained model instead. Tip: restart the kernel and run all cells in order to try again.")

if not trained_live:
    model = keras.models.load_model(locate("churn_ann_model.h5"), compile=False)
    print("Loaded the saved, already-trained model.")
    curves = locate("accuracy_loss_graph.png", required=False)
    if curves:
        display(Image(filename=str(curves), width=800))

## 5. Evaluation on 1,409 unseen customers

In [ ]:
X_test, y_test = test[FEATURES].values.astype("float32"), test.Churn.values
p_test = model.predict(X_test, verbose=0).ravel()
p_val  = model.predict(X_val,  verbose=0).ravel()

def metrics(y, p, thr):
    yh = (p >= thr).astype(int)
    return {"Accuracy": accuracy_score(y, yh), "Precision": precision_score(y, yh), "Recall": recall_score(y, yh),
            "F1": f1_score(y, yh), "AUC": roc_auc_score(y, p)}

# the tuned threshold is chosen on VALIDATION data, never on the test set
grid = np.arange(0.10, 0.90, 0.01)
THR = float(grid[int(np.argmax([f1_score(y_val, p_val >= t) for t in grid]))])

lr = LogisticRegression(max_iter=1000).fit(train[FEATURES], train.Churn)
table = pd.DataFrame({"ANN, threshold 0.50": metrics(y_test, p_test, 0.5),
                      f"ANN, tuned threshold {THR:.2f}": metrics(y_test, p_test, THR),
                      "Logistic regression (baseline)": metrics(y_test, lr.predict_proba(test[FEATURES])[:, 1], 0.5)}).T
display(fmt(table, **{c: "{:.1%}" for c in table.columns}))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
cm = confusion_matrix(y_test, (p_test >= 0.5).astype(int))
ConfusionMatrixDisplay(cm, display_labels=["No churn", "Churn"]).plot(ax=ax[0], cmap="Blues", colorbar=False)
ax[0].set_title("Confusion matrix (threshold 0.50)")
fpr, tpr, _ = roc_curve(y_test, p_test)
ax[1].plot(fpr, tpr, label=f"ANN (AUC = {roc_auc_score(y_test, p_test):.3f})"); ax[1].plot([0, 1], [0, 1], "k--", label="Random guess")
ax[1].set(title="ROC curve", xlabel="False positive rate", ylabel="True positive rate"); ax[1].legend(loc="lower right"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()
print(f"Of {int(y_test.sum())} customers who really churned, the model caught {cm[1, 1]} at threshold 0.50.")

## 6. Predicting churn for individual customers
The model gives each customer a churn probability. Above the threshold, we flag them for retention action.

In [ ]:
res = test[["Churn"]].copy()
res["churn_probability"] = p_test
res["predicted"] = (p_test >= THR).astype(int)
res["tenure_months"]  = (test.tenure * raw.tenure.std(ddof=0) + raw.tenure.mean()).round().astype(int)
res["monthly_charge"] = (test.MonthlyCharges * raw.MonthlyCharges.std(ddof=0) + raw.MonthlyCharges.mean()).round(2)
res["senior"] = test.SeniorCitizen.map({0: "No", 1: "Yes"}); res["dependents"] = test.Dependents.map({0: "No", 1: "Yes"})
res["segment"] = pd.Series(km.predict(test[["tenure", "MonthlyCharges"]].values)).map(seg_name).values   # which customer segment

picks = {"Highest risk": res.churn_probability.idxmax(),
         "Close to the threshold": (res.churn_probability - THR).abs().idxmin(),
         "Lowest risk": res.churn_probability.idxmin()}
sample = res.loc[list(picks.values())].copy()
sample.insert(0, "customer", list(picks.keys()))
sample["decision"] = np.where(sample.predicted == 1, "Contact", "No action")
sample["actually_churned"] = sample.Churn.map({1: "Yes", 0: "No"})
display(fmt(sample[["customer", "tenure_months", "monthly_charge", "senior", "dependents",
                    "churn_probability", "decision", "actually_churned"]].set_index("customer"),
            churn_probability="{:.1%}", monthly_charge="${:.2f}"))

In [ ]:
# Segment-level view: where is the model most useful?
def seg_stats(d):
    flagged = d.predicted == 1
    return pd.Series({"test customers": len(d), "actual churn": d.Churn.mean(), "flagged by model": flagged.mean(),
                      "churners caught": flagged[d.Churn == 1].mean() if (d.Churn == 1).any() else np.nan})
seg_view = res.groupby("segment").apply(seg_stats).sort_values("actual churn", ascending=False)
display(fmt(seg_view, **{"test customers": "{:.0f}", "actual churn": "{:.1%}", "flagged by model": "{:.1%}", "churners caught": "{:.1%}"}))

## 7. What drives churn?
Saved results from `churn_drivers.py` (each factor shuffled 30 times to measure how much the model relies on it).

In [ ]:
imp_file = locate("feature_importance.csv", required=False)
if imp_file:
    imp = pd.read_csv(imp_file)[["factor", "auc_drop"]]
    display(imp.rename(columns={"auc_drop": "drop in AUC when shuffled"}).set_index("factor").round(3))
else:
    print("feature_importance.csv not found. Run churn_drivers.py first.")
for f in ["driver_importance.png", "churn_heatmap_tenure_charges.png"]:
    img_path = locate(f, required=False)
    if img_path:
        display(Image(filename=str(img_path), width=620))

## 8. Recommendation in action: a retention call list
First-year customers the model flags as likely to leave, highest risk first.

In [ ]:
call_list = (res[(res.tenure_months <= 12) & (res.predicted == 1)]
             .sort_values("churn_probability", ascending=False).head(10)
             [["segment", "tenure_months", "monthly_charge", "senior", "dependents", "churn_probability"]])
display(fmt(call_list, churn_probability="{:.1%}", monthly_charge="${:.2f}"))
n_first = int(((res.tenure_months <= 12) & (res.predicted == 1)).sum())
print(f"{n_first} first-year customers in the test set are flagged. A monthly refresh of this list tells the retention team who to contact.")

## 9. Limitations (checks run in `limitations_checks.py`)

In [ ]:
lim_file = locate("limitations_check_results.csv", required=False)
if lim_file:
    display(pd.read_csv(lim_file))
else:
    print("limitations_check_results.csv not found. Run limitations_checks.py first.")

## Summary
- Churn is concentrated among **new customers paying more**: the New Premium segment churns at 49% and produces 58% of all churners.
- The ANN reaches an AUC of about **0.80** and catches about **64%** of leavers at the tuned threshold.
- The best next step is to act on first-year customers while improving the data, especially for loyal customers.